# Exploring the importance of model weight norm to grokking in small transformers learning modular addition

This notebook reproduces Figure 7 from the paper [Omnigrok: Grokking Beyond Algorithmic Data](https://arxiv.org/abs/2210.01117), showing how constraining weight norm influences grokking. The setup here is similar to the original paper on grokking from [Power et al.](https://arxiv.org/abs/2201.02177) -- we train small transformer models to do math operations, here modular addition. We use exactly the same setup as Nanda and Lieberum's [A Mechanistic Interpretability Analysis of Grokking](https://www.alignmentforum.org/posts/N6WM6hs7RQMKDhYjB/a-mechanistic-interpretability-analysis-of-grokking). The transformer implementation and the choice of hyperparameters is taken directly from [Neel's colab notebook](https://colab.research.google.com/drive/1F6_1_cWXE5M7WocUcpQWp3v8z4b1jL20). We'll see that constraining optimization to lie on a hypersphere of constant model weight norm significantly changes the learning dynamics, bringing train and test accuracy curves together.

### Imports

In [ ]:
!nvidia-smi

In [ ]:
!pip install einops

In [ ]:
!pip install matplotlib

In [14]:
import os
import random
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import einops
from tqdm.auto import tqdm
from collections import OrderedDict

import matplotlib.pyplot as plt

### Transformer implementation

In [15]:
# This code was taken directly from Neel Nanda's study of grokking:
# https://colab.research.google.com/drive/1F6_1_cWXE5M7WocUcpQWp3v8z4b1jL20

class HookPoint(nn.Module):
    def __init__(self):
        super().__init__()
        self.fwd_hooks = []
        self.bwd_hooks = []
    
    def give_name(self, name):
        # Called by the model at initialisation
        self.name = name
    
    def add_hook(self, hook, dir='fwd'):
        # Hook format is fn(activation, hook_name)
        # Change it into PyTorch hook format (this includes input and output, 
        # which are the same for a HookPoint)
        def full_hook(module, module_input, module_output):
            return hook(module_output, name=self.name)
        if dir=='fwd':
            handle = self.register_forward_hook(full_hook)
            self.fwd_hooks.append(handle)
        elif dir=='bwd':
            handle = self.register_backward_hook(full_hook)
            self.bwd_hooks.append(handle)
        else:
            raise ValueError(f"Invalid direction {dir}")
    
    def remove_hooks(self, dir='fwd'):
        if (dir=='fwd') or (dir=='both'):
            for hook in self.fwd_hooks:
                hook.remove()
            self.fwd_hooks = []
        if (dir=='bwd') or (dir=='both'):
            for hook in self.bwd_hooks:
                hook.remove()
            self.bwd_hooks = []
        if dir not in ['fwd', 'bwd', 'both']:
            raise ValueError(f"Invalid direction {dir}")
    
    def forward(self, x):
        return x


In [92]:
# Embed & Unembed
class Embed(nn.Module):
    def __init__(self, d_vocab, d_model):
        super().__init__()
        self.W_E = nn.Parameter(torch.randn(d_model, d_vocab)/np.sqrt(d_model))
    
    def forward(self, x):
        return torch.einsum('dbp -> bpd', self.W_E[:, x])

class Unembed(nn.Module):
    def __init__(self, d_vocab, d_model):
        super().__init__()
        self.W_U = nn.Parameter(torch.randn(d_model, d_vocab)/np.sqrt(d_vocab))
    
    def forward(self, x):
        return (x @ self.W_U)

# Positional Embeddings
class PosEmbed(nn.Module):
    def __init__(self, max_ctx, d_model):
        super().__init__()
        self.W_pos = nn.Parameter(torch.randn(max_ctx, d_model)/np.sqrt(d_model))
    
    def forward(self, x):
        return x+self.W_pos[:x.shape[-2]]

# RMSNorm
class RMSNorm(nn.Module):
    def __init__(self, d_model, epsilon = 1e-4, model=[None]):
        super().__init__()
        self.model = model
        self.w_ln = nn.Parameter(torch.ones(d_model))
        self.b_ln = nn.Parameter(torch.zeros(d_model))
        self.epsilon = epsilon
    
    def forward(self, x):
        if self.model[0].use_ln:
            # Compute root mean square over the last dimension
            rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.epsilon)
            x = x / rms
            x = x * self.w_rms
            x = x + self.b_rms
            return x
        else:
            return x
# Attention
class Attention(nn.Module):
    def __init__(self, d_model, num_heads, d_v, d_k, n_ctx, model):
        super().__init__()
        self.model = model
        self.W_K = nn.Parameter(torch.randn(num_heads, d_k, d_model)/np.sqrt(d_model))
        self.W_Q = nn.Parameter(torch.randn(num_heads, d_k, d_model)/np.sqrt(d_model))
        self.W_V = nn.Parameter(torch.randn(num_heads, d_v, d_model)/np.sqrt(d_model))
        self.W_O = nn.Parameter(torch.randn(d_model, d_v * num_heads)/np.sqrt(d_model))
        self.register_buffer('mask', torch.tril(torch.ones((n_ctx, n_ctx))))
        self.d_head = d_k
        self.hook_k = HookPoint()
        self.hook_q = HookPoint()
        self.hook_v = HookPoint()
        self.hook_z = HookPoint()
        self.hook_attn = HookPoint()
        self.hook_attn_pre = HookPoint()

    def forward(self, x):
        k = self.hook_k(torch.einsum('ihd,bpd->biph', self.W_K, x))
        q = self.hook_q(torch.einsum('ihd,bpd->biph', self.W_Q, x))
        v = self.hook_v(torch.einsum('ihd,bpd->biph', self.W_V, x))
        attn_scores_pre = torch.einsum('biph,biqh->biqp', k, q)
        attn_scores_masked = torch.tril(attn_scores_pre) - 1e10 * (1 - self.mask[:x.shape[-2], :x.shape[-2]])
        attn_matrix = self.hook_attn(F.softmax(self.hook_attn_pre(attn_scores_masked/np.sqrt(self.d_head)), dim=-1))
        z = self.hook_z(torch.einsum('biph,biqp->biqh', v, attn_matrix))
        z_flat = einops.rearrange(z, 'b i q h -> b q (i h)')
        out = torch.einsum('df,bqf->bqd', self.W_O, z_flat)
        return out

# MLP Layers
class MLP(nn.Module):
    def __init__(self, d_model, d_mlp, act_type, model):
        super().__init__()
        self.model = model
        self.W_in = nn.Parameter(torch.randn(d_mlp, d_model)/np.sqrt(d_model))
        self.b_in = nn.Parameter(torch.zeros(d_mlp))
        self.W_out = nn.Parameter(torch.randn(d_model, d_mlp)/np.sqrt(d_model))
        self.b_out = nn.Parameter(torch.zeros(d_model))
        self.act_type = act_type
        # self.ln = LayerNorm(d_mlp, model=self.model)
        self.hook_pre = HookPoint()
        self.hook_post = HookPoint()
        assert act_type in ['ReLU', 'GeLU']
        
    def forward(self, x):
        x = torch.einsum('md,bpd->bpm', self.W_in, x) + self.b_in
        
        if self.act_type=='ReLU':
            x = F.relu(x)
        elif self.act_type=='GeLU':
            x = F.gelu(x)
        x = self.hook_post(x)
        x = torch.einsum('dm,bpm->bpd', self.W_out, x) + self.b_out
        return x

# Transformer Block
class TransformerBlock(nn.Module):
    def __init__(self, d_model, d_mlp, d_v, d_k, num_heads, n_ctx, act_type, model):
        super().__init__()
        self.model = model
        # self.ln1 = LayerNorm(d_model, model=self.model)
        self.attn = Attention(d_model, num_heads, d_v, d_k, n_ctx, model=self.model)
        # self.ln2 = LayerNorm(d_model, model=self.model)
        self.mlp = MLP(d_model, d_mlp, act_type, model=self.model)
        self.hook_attn_out = HookPoint()
        self.hook_mlp_out = HookPoint()
        self.hook_resid_pre = HookPoint()
        self.hook_resid_mid = HookPoint()
        self.hook_resid_post = HookPoint()
    
    def forward(self, x):
        x = self.hook_resid_mid(x + self.hook_attn_out(self.attn((self.hook_resid_pre(x)))))
        x = self.hook_resid_post(x + self.hook_mlp_out(self.mlp((x))))
        return x

# Full transformer
class Transformer(nn.Module):
    def __init__(self, num_layers, d_vocab, d_model, d_mlp, d_v, d_k, num_heads, n_ctx, act_type, use_cache=False, use_ln=True):
        super().__init__()
        self.cache = {}
        self.use_cache = use_cache

        self.embed = Embed(d_vocab, d_model)
        self.pos_embed = PosEmbed(n_ctx, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, d_mlp, d_v, d_k, num_heads, n_ctx, act_type, model=[self]) for i in range(num_layers)])
        # self.ln = LayerNorm(d_model, model=[self])
        self.unembed = Unembed(d_vocab, d_model)
        self.use_ln = use_ln

        for name, module in self.named_modules():
            if type(module)==HookPoint:
                module.give_name(name)
    
    def forward(self, x):
        x = self.embed(x)
        x = self.pos_embed(x)
        
        for block in self.blocks:
            x = block(x)
        # x = self.ln(x)
        x = self.unembed(x)
        return x[:, -1]

    def set_use_cache(self, use_cache):
        self.use_cache = use_cache
    
    def hook_points(self):
        return [module for name, module in self.named_modules() if 'hook' in name]

    def remove_all_hooks(self):
        for hp in self.hook_points():
            hp.remove_hooks('fwd')
            hp.remove_hooks('bwd')
    
    def cache_all(self, cache, incl_bwd=False):
        # Caches all activations wrapped in a HookPoint
        def save_hook(tensor, name):
            cache[name] = tensor.detach()
        def save_hook_back(tensor, name):
            cache[name+'_grad'] = tensor[0].detach()
        for hp in self.hook_points():
            hp.add_hook(save_hook, 'fwd')
            if incl_bwd:
                hp.add_hook(save_hook_back, 'bwd')

## Composable function-preserving Expansions for Transformer Architecture

In [93]:
def params_pad_to_shape(source_state, target_state, function_preserving=True):

    final_state = OrderedDict()    
    for name, target_tensor in target_state.items():
        # print(f"{name}, shape: {target_tensor.shape}")

        if name in source_state:
            source_tensor = source_state[name]
            # print(f"Found in source state dict, shape: {source_tensor.shape}, sum: {source_tensor.sum().item()}" )

            assert len(source_tensor.shape) == len(target_tensor.shape)

            to_pad_shape = []
            for dim in range(len(target_tensor.shape)):
                dim_diff = target_tensor.shape[dim] - source_tensor.shape[dim]
                to_pad_shape.append([0, dim_diff])

            # in pytorch the dimensions needs to be reversed
            to_pad_shape = to_pad_shape[::-1]
            # print("to pad shape:", to_pad_shape)

            zero_init = False
            head_pad = 0
            s_n_head = 0
            # MLP expansion Sec 3.1
            if "W_out" in name and to_pad_shape[0][1] > 0:
                # print(f"Expanding MLP: {name}")
                zero_init = True

            # Heads addition Sec 3.2 or Head expansion Sec 3.3
            if ("W_O" in name and to_pad_shape[0][1] > 0):
                name_of_val_param = name[:-1] + "V" # W_V for the W_O
                # check if W_V dim has changed at dim=1
                t_n_head, t_d_head, t_d_model = target_state[name_of_val_param].shape
                sx_n_head, s_d_head, s_d_model = source_state[name_of_val_param].shape
                head_pad = t_d_head - s_d_head
                s_n_head = sx_n_head
                zero_init = (t_n_head - s_n_head) > 0
                to_pad_shape[0][1] -= s_n_head * head_pad
                # print(f"Updated zero padding shape {to_pad_shape}")


            # Attention expansion Sec 3.4
            if "W_K" in name and to_pad_shape[1][1] > 0:
                # print(f"Expainding attention W_k: {name}")
                zero_init = True
                
            # Hidden dimension expansion Sec 3.5
            if ("W_out" in name and to_pad_shape[-1][1] > 0) or \
                ("b_out" in name and to_pad_shape[-1][1] > 0) or \
                ("W_O" in name and to_pad_shape[1][1] > 0) or \
                ("W_pos" in name and to_pad_shape[0][1] > 0) or \
                ("W_E" in name and to_pad_shape[-1][1] > 0) or \
                ("W_U" in name and to_pad_shape[0][1] > 0):
                # print(f"Expainding hidden dimension param: {name}")
                zero_init = True

             # Key matrix scaling for attentin expansion Sec 3.4
            if "W_K" in name and to_pad_shape[1][1] > 0:
                source_tensor = source_tensor * torch.sqrt(
                    torch.tensor(target_tensor.shape[1] / source_tensor.shape[1])
                )

            # norm scaling for attentino expansion Sec 3.4
            if "w_ln" in name and to_pad_shape[-1][1] > 0:
                raise NotImplementedError("The Norms are skipped in OmniGROK check implementation!")

            if head_pad:
                # we update the source_tensor to match the requried d_head'
                # print("padding head")
                source_tensor = source_tensor.view(source_tensor.shape[0], s_n_head, -1)
                pad = torch.randn((source_tensor.shape[0], s_n_head, head_pad),
                                  dtype=source_tensor.dtype, device=source_tensor.device) * 1e-9
                source_tensor = torch.cat([source_tensor, pad], dim=2).view(source_tensor.shape[0], -1)
                
            if zero_init:
                # print("0 init")
                padded_tensor = F.pad(
                    source_tensor, [p for pair in to_pad_shape for p in pair],
                    "constant", value=0
                )                # Create noise tensor and mask only the padded region
                noise = torch.randn_like(padded_tensor) * 1e-8
                mask = (padded_tensor == 0) & (F.pad(source_tensor, [p for pair in to_pad_shape for p in pair], value=float('nan')).isnan())

                padded_tensor += noise * mask  # Add noise only to padded parts
            else:
                # print("No 0 init")
                # Start from target_tensor's random values
                padded_tensor = target_tensor.clone()

                # Copy source tensor
                slices = tuple(slice(0, s) for s in source_tensor.shape)
                padded_tensor[slices] = source_tensor
            
            final_state[name] = padded_tensor
        
        else: # if param not in source tensor
            # print("Not found in source state dict")s
            # Layer addition Sec 3.6
            if "W_O" in name or "W_out" in name:
                # print("0 init")
                padded_tensor = torch.randn_like(target_tensor, dtype=target_tensor.dtype, device=target_tensor.device) * 1e-9
            else:
                # print("No 0 init")
                padded_tensor =  target_tensor.clone()

            final_state[name] = padded_tensor

        # print("-----")
    return final_state
            

##### MLP expansion (Section 3.1)

In [94]:
ORIGINAL_p = 6
ORIGINAL_h = 5
BATCH = 3
SEQUENCE = 2

mlp = MLP(d_model=ORIGINAL_h, d_mlp=ORIGINAL_p, act_type="ReLU", model=None)

MOD_p = ORIGINAL_p + 3
MOD_mlp = MLP(d_model=ORIGINAL_h, d_mlp=MOD_p, act_type="ReLU", model=None)

MOD_mlp.load_state_dict(
    params_pad_to_shape(mlp.state_dict(), MOD_mlp.state_dict())
    )


X_mlp = torch.randn(BATCH, SEQUENCE, ORIGINAL_h)

print("Original:",  mlp(X_mlp).sum().item(), "Expanded MLP:", MOD_mlp(X_mlp).sum().item())
assert torch.allclose(mlp(X_mlp), MOD_mlp(X_mlp))

Original: 0.35897270347753546 Expanded MLP: 0.3589726774017846


#### Head addition on MHA (Section 3.2)

In [95]:
# expecting d_k = d_v
ORIGINAL_k = 6
ORIGINAL_v = 6

ORIGINAL_h = 8
ORIGINAL_E = 2
BATCH = 3
SEQUENCE = 2

mha = Attention(d_model=ORIGINAL_h, num_heads=ORIGINAL_E, d_k=ORIGINAL_k, d_v=ORIGINAL_v, n_ctx=3, model=None)

MOD_E = ORIGINAL_E + 3

MOD_mha = Attention(d_model=ORIGINAL_h, num_heads=MOD_E, d_k=ORIGINAL_k, d_v=ORIGINAL_v, n_ctx=3, model=None)

MOD_mha.load_state_dict(
    params_pad_to_shape(mha.state_dict(), MOD_mha.state_dict())
    )

X_MHA = torch.randn(BATCH, SEQUENCE, ORIGINAL_h)

print("Original:",  mha(X_MHA).sum().item(), "Added Head:", MOD_mha(X_MHA).sum().item())
assert torch.allclose(mha(X_MHA), MOD_mha(X_MHA))

Original: -6.550595132463287 Added Head: -6.5505949979128815


#### Heads expansion on MHA (Section 3.3)

In [96]:
# expecting d_k = d_v
ORIGINAL_k = 6
ORIGINAL_v = 6

ORIGINAL_h = 5
ORIGINAL_E = 2
BATCH = 3
SEQUENCE = 2

mha = Attention(d_model=ORIGINAL_h, num_heads=ORIGINAL_E, d_k=ORIGINAL_k, d_v=ORIGINAL_v, n_ctx=3, model=None)

MOD_v = ORIGINAL_v + 3
MOD_mha = Attention(d_model=ORIGINAL_h, num_heads=ORIGINAL_E+1, d_k=ORIGINAL_k, d_v=MOD_v, n_ctx=3, model=None)

MOD_mha.load_state_dict(
    params_pad_to_shape(mha.state_dict(), MOD_mha.state_dict())
    )

X_MHA = torch.randn(BATCH, SEQUENCE, ORIGINAL_h)

print("Original:",  mha(X_MHA).sum().item(), "Expanded heads:", MOD_mha(X_MHA).sum().item())
assert torch.allclose(mha(X_MHA), MOD_mha(X_MHA))

Original: 0.930768003078775 Expanded heads: 0.9307677578930464


#### Attention Expansion on MHA (Section 3.4)

In [97]:
# expecting d_k = d_v
ORIGINAL_k = 6
ORIGINAL_v = 6

ORIGINAL_h = 5
ORIGINAL_E = 2
BATCH = 3
SEQUENCE = 2

mha = Attention(d_model=ORIGINAL_h, num_heads=ORIGINAL_E, d_k=ORIGINAL_k, d_v=ORIGINAL_v, n_ctx=3, model=None)

MOD_k = ORIGINAL_k + 3
MOD_mha = Attention(d_model=ORIGINAL_h, num_heads=ORIGINAL_E, d_k=MOD_k, d_v=ORIGINAL_v, n_ctx=3, model=None)

MOD_mha.load_state_dict(
    params_pad_to_shape(mha.state_dict(), MOD_mha.state_dict())
    )

X_MHA = torch.randn(BATCH, SEQUENCE, ORIGINAL_h)

print("Original:",  mha(X_MHA).sum().item(), "Expanded heads:", MOD_mha(X_MHA).sum().item())
assert torch.allclose(mha(X_MHA), MOD_mha(X_MHA),rtol=1e-5)

Original: 9.4983001151639 Expanded heads: 9.498300135364385


#### MHA Combination

In [98]:
# expecting d_k = d_v
ORIGINAL_k = 6
ORIGINAL_v = 6

ORIGINAL_h = 5
ORIGINAL_E = 2
BATCH = 3
SEQUENCE = 2

mha = Attention(d_model=ORIGINAL_h, num_heads=ORIGINAL_E, d_k=ORIGINAL_k, d_v=ORIGINAL_v, n_ctx=3, model=None)

MOD_k = ORIGINAL_k + 3
MOD_v = ORIGINAL_v + 3
MOD_E = ORIGINAL_E + 2

MOD_mha = Attention(d_model=ORIGINAL_h, num_heads=MOD_E, d_k=MOD_k, d_v=MOD_v, n_ctx=3, model=None)

MOD_mha.load_state_dict(
    params_pad_to_shape(mha.state_dict(), MOD_mha.state_dict())
    )

X_MHA = torch.randn(BATCH, SEQUENCE, ORIGINAL_h)

print("Original:",  mha(X_MHA).sum().item(), "Expanded heads:", MOD_mha(X_MHA).sum().item())
assert torch.allclose(mha(X_MHA), MOD_mha(X_MHA),rtol=1e-5)

Original: -4.48683626485793 Expanded heads: -4.486836433223308


#### Hidden Dimension Expansion (Section 3.5)

In [99]:
ORIGINAL_k = 6
ORIGINAL_v = 6
ORIGINAL_h = 5
ORIGINAL_E = 2
ORIGINAL_p = 4
BATCH = 3
SEQUENCE = 2


layer = TransformerBlock(d_model=ORIGINAL_h, d_mlp=ORIGINAL_p, d_v=ORIGINAL_v, d_k=ORIGINAL_k, num_heads=ORIGINAL_E, n_ctx=3, 
                 act_type="ReLU", model=None)
MOD_h = ORIGINAL_h + 2


I_layer = torch.randn(BATCH, SEQUENCE, ORIGINAL_h)
O_layer = layer(I_layer)
O_layer_padded = F.pad(O_layer, (0, MOD_h - ORIGINAL_h), 'constant', value=0)


MOD_layer = TransformerBlock(d_model=MOD_h, d_mlp=ORIGINAL_p, d_v=ORIGINAL_v, d_k=ORIGINAL_k, num_heads=ORIGINAL_E, n_ctx=3, 
                 act_type="ReLU", model=None)
MOD_I_layer = F.pad(I_layer, (0, MOD_h - ORIGINAL_h), 'constant', value=0)

MOD_layer.load_state_dict(
    params_pad_to_shape(layer.state_dict(), MOD_layer.state_dict())
    )
MOD_O_layer = MOD_layer(MOD_I_layer)

print("Original:",  O_layer_padded.sum().item(), "Expanded hidden dimension:", MOD_O_layer.sum().item())
assert torch.allclose(O_layer_padded, MOD_O_layer, atol=1e-5)

Original: -19.24045747316 Expanded hidden dimension: -19.240457253540455


#### Layer Addition (Section 3.6)

In [100]:
ORIGINAL_k = 6
ORIGINAL_v = 6
ORIGINAL_h = 5
ORIGINAL_E = 2
ORIGINAL_p = 4
ORIGINAL_N = 2
BATCH = 3
SEQUENCE = 2
HOUT = 2

model = Transformer(num_layers=ORIGINAL_N, d_vocab=113, d_model=ORIGINAL_h, d_mlp=ORIGINAL_p, 
                d_v=ORIGINAL_v, d_k=ORIGINAL_k, num_heads=ORIGINAL_E, n_ctx=3, 
                act_type="ReLU")

X = torch.randint(113,(BATCH, SEQUENCE))
o = model(X)

MOD_N = ORIGINAL_N + 3
MOD_model = Transformer(num_layers=MOD_N, d_vocab=113, d_model=ORIGINAL_h, d_mlp=ORIGINAL_p, 
                    d_v=ORIGINAL_v, d_k=ORIGINAL_k, num_heads=ORIGINAL_E, n_ctx=3, 
                    act_type="ReLU")


MOD_model.load_state_dict(
    params_pad_to_shape(model.state_dict(), MOD_model.state_dict())
    )

MOD_o = MOD_model(X)

print("Original:",  o.sum().item(), "Added layers:", MOD_o.sum().item())
assert torch.allclose(o, MOD_o)

Original: -13.375525114871284 Added layers: -13.37552515417616


#### Hidden Dimension Expansion (Sec. 3.5) on full model

In [101]:
ORIGINAL_k = 6
ORIGINAL_v = 6
ORIGINAL_h = 5
ORIGINAL_E = 2
ORIGINAL_p = 4
ORIGINAL_N = 2
BATCH = 3
SEQUENCE = 2
HOUT = 2

model = Transformer(num_layers=ORIGINAL_N, d_vocab=11, d_model=ORIGINAL_h, d_mlp=ORIGINAL_p, 
                d_v=ORIGINAL_v, d_k=ORIGINAL_k, num_heads=ORIGINAL_E, n_ctx=3, 
                act_type="ReLU")

X = torch.randint(11,(BATCH, SEQUENCE))
o = model(X)

MOD_h = ORIGINAL_h + 3


# o_pad = F.pad(o, (0, MOD_h - ORIGINAL_h), 'constant', value=0)
MOD_model = Transformer(num_layers=ORIGINAL_N, d_vocab=11, d_model=MOD_h, d_mlp=ORIGINAL_p, 
                    d_v=ORIGINAL_v, d_k=ORIGINAL_k, num_heads=ORIGINAL_E, n_ctx=3, 
                    act_type="ReLU")


MOD_model.load_state_dict(
    params_pad_to_shape(model.state_dict(), MOD_model.state_dict())
    )

MOD_o = MOD_model(X)

print("Original:",  o.sum().item(), "Expanded hidden dimension:", MOD_o.sum().item())
assert torch.allclose(o, MOD_o)

Original: 7.312271960536885 Expanded hidden dimension: 7.312272641803206


#### All transformations on Full Model

In [102]:
ORIGINAL_k = 6
ORIGINAL_v = 6
ORIGINAL_h = 5
ORIGINAL_E = 2
ORIGINAL_p = 4
ORIGINAL_N = 2
BATCH = 3
SEQUENCE = 2
HOUT = 2

model = Transformer(num_layers=ORIGINAL_N, d_vocab=11, d_model=ORIGINAL_h, d_mlp=ORIGINAL_p, 
                d_v=ORIGINAL_v, d_k=ORIGINAL_k, num_heads=ORIGINAL_E, n_ctx=3, 
                act_type="ReLU")

X = torch.randint(11,(BATCH, SEQUENCE))
o = model(X)

MOD_h = ORIGINAL_h + 3
MOD_N = ORIGINAL_N + 2
MOD_p = ORIGINAL_p + 2
MOD_k = ORIGINAL_k + 4
MOD_v = ORIGINAL_v + 2
MOD_E = ORIGINAL_E + 1

MOD_model = Transformer(num_layers=MOD_N, d_vocab=11, d_model=MOD_h, d_mlp=MOD_p, 
                    d_v=MOD_v, d_k=MOD_k, num_heads=MOD_E, n_ctx=3, 
                    act_type="ReLU")


MOD_model.load_state_dict(
    params_pad_to_shape(model.state_dict(), MOD_model.state_dict())
    )

MOD_o = MOD_model(X)

print("Original:",  o.sum().item(), "Expanded hidden dimension:", MOD_o.sum().item())
assert torch.allclose(o, MOD_o, rtol=1e-4)

Original: 6.37426369145998 Expanded hidden dimension: 6.374264129050736


### Baseline training run

In [177]:
def full_loss(model, data, device):
    loader = torch.utils.data.DataLoader(data, batch_size=len(data), shuffle=False)
    # Take the final position only
    x, labels = next(iter(loader))
    x = x.to(device)
    labels = labels.to(device)
    logits = model(x)# [:, -1]
    return torch.nn.functional.cross_entropy(logits, labels)

# def full_loss(model, data, device):
#     loader = torch.utils.data.DataLoader(data, batch_size=len(data), shuffle=False)
#     # Take the final position only
#     x, labels = next(iter(loader))
#     x = x.to(device)
#     labels = labels.to(device)
#     logits = model(x)# [:, -1]
#     return torch.nn.functional.cross_entropy(logits, labels)

def full_accuracy(model, data, device):
    loader = torch.utils.data.DataLoader(data, batch_size=len(data), shuffle=False)
    # Take the final position only
    x, labels = next(iter(loader))
    x = x.to(device)
    labels = labels.to(device)
    logits = model(x)# [:, -1]
    predictions = torch.argmax(logits, dim=1)
    return torch.sum(predictions == labels).item() / len(labels)

In [130]:
seed = 42
p = 113
fraction = 0.3

In [132]:
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.backends.cudnn.deterministic = True

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
torch.set_default_dtype(torch.float64)
print(f"Using {device}")

Using cuda:0


In [134]:
equals_token = p
x, y = torch.meshgrid(torch.arange(p), torch.arange(p), indexing='ij')
x = x.flatten()
y = y.flatten()
# plus = torch.ones(x.shape, dtype=torch.int64) * plus_token
equals = torch.ones(x.shape, dtype=torch.int64) * equals_token
prompts = torch.stack([x, y, equals], dim=1).to(device)
answers = ((x + y) % p).to(device)

data = torch.utils.data.TensorDataset(prompts, answers)
train, test = torch.utils.data.random_split(data, 
                                [int(fraction * len(data)),
                                len(data) - int(fraction * len(data))
                                ])

modelx = Transformer(num_layers=1, 
                    d_vocab=equals_token+1, 
                    d_model=128,
                    d_mlp=512,
                    d_k=32,
                    d_v=32,
                    num_heads=4,
                    n_ctx=3, # context length
                    act_type='ReLU', 
                    use_cache=False, 
                    use_ln=False # use LayerNorm
                ).to(device)

In [136]:
from dataclasses import dataclass

@dataclass
class ModelSpec:
    d_model: int
    d_mlp: int
    n_head: int

    def __post_init__(self):
        assert self.d_model % self.n_head == 0, \
            f"d_model ({self.d_model}) must be divisible by n_head ({self.n_head})"

    @property
    def d_head(self):
        return self.d_model // self.n_head

    def __repr__(self):
        return (f"ModelSpec(d_model={self.d_model}, "
            f"d_mlp={self.d_mlp}, "
            f"n_head={self.n_head}, "
            f"d_head={self.d_head})")

In [138]:
from pyhessian import hessian
import copy

In [178]:

def make_loss_fn(data, device):
    def loss_fn(model):
        return full_loss(model, data, device)

    return loss_fn

def get_params(model_orig, model_perb, direction, alpha):
    for m_orig, m_perb, d in zip(model_orig.parameters(), model_perb.parameters(), direction):
        m_perb.data = m_orig.data + alpha * d
    return model_perb


In [179]:
from datetime import datetime
import hashlib
import json

def train_and_plot(sizes, t_steps=5000, exp_freq=None, log_dir="./logs/"):
    log_steps = []
    train_losses = []
    test_losses = []
    train_accuracies = []
    test_accuracies = []
    norms = []
    grad_norms = []

    # if not exp_freq:
    #     exp_freq = t_steps // len(sizes)
    size_index = 0
    model = Transformer(num_layers=1, 
                        d_vocab=equals_token+1, 
                        d_model=sizes[0].d_model,
                        d_mlp=sizes[0].d_mlp,
                        d_k=sizes[0].d_head,
                        d_v=sizes[0].d_head,
                        num_heads=sizes[0].n_head,
                        n_ctx=3, # context length
                        act_type='ReLU', 
                        use_cache=False, 
                        use_ln=False # use LayerNorm
                    ).to(device)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1.0, betas=(0.9, 0.98))
        
    for epoch in tqdm(range(t_steps)):
        # if epoch % exp_freq == 0:
        if len(test_accuracies) and test_accuracies[-1] > 0.25 and size_index == 0:
            # idx = epoch // exp_freq
            size_index = 1 
            exp_freq = epoch
            if size_index > 0 and size_index < len(sizes):
                print( f"@epoch {epoch}\t{sizes[size_index -1]} -> {sizes[size_index]}")
                
                model_2 = Transformer(num_layers=1, 
                            d_vocab=equals_token+1, 
                            d_model=sizes[size_index].d_model,
                            d_mlp=sizes[size_index].d_mlp,
                            d_k=sizes[size_index].d_head,
                            d_v=sizes[size_index].d_head,
                            num_heads=sizes[size_index].n_head,
                            n_ctx=3, # context length
                            act_type='ReLU', 
                            use_cache=False, 
                            use_ln=False # use LayerNorm
                        ).to(device)
                model_2.load_state_dict(
                    params_pad_to_shape(model.state_dict(), model_2.state_dict())
                )
                model = model_2
                optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1.0, betas=(0.9, 0.98))
            
        train_loss = full_loss(model, train, device)
        if epoch % 30 == 0:
            with torch.no_grad():
                log_steps.append(epoch)
                test_loss = full_loss(model, test, device)
                train_losses.append(train_loss.item())
                test_losses.append(test_loss.item())
                train_accuracies.append(full_accuracy(model, train, device))
                test_accuracies.append(full_accuracy(model, test, device))
                norms.append(np.sqrt(sum(param.pow(2).sum().item() for param in model.parameters())))
                

            # Loss landscape with Hessian of layers
            
            # load loss function
            # loss_fn = make_loss_fn(train, device)
            criterion = torch.nn.functional.cross_entropy
            train_loader = torch.utils.data.DataLoader(train, batch_size=len(train), shuffle=False)
            
            hessian_comp = hessian(model=model, criterion=criterion, cuda=True, dataloader=train_loader)

            top_eigenvalues, top_eigenvector = hessian_comp.eigenvalues()

            # saving top eigenvalues
            eigenvalues_np = np.array(top_eigenvalues)
            np.save(os.path.join(log_dir + "/hessian_eigenvalues", f"eigvals_epoch{epoch}.npy"), eigenvalues_np)

            lams = np.linspace(-0.5, 0.5, 21).astype(np.float32)

            loss_list = []

            # computing and saving loss landscape plots
            model_perb = copy.deepcopy(model).to(device)
            for lam in lams:
                model_perb = get_params(model, model_perb, top_eigenvector[0], lam)
                loss_val = full_loss(model_perb, train, device)
                loss_list.append(loss_val.item())

            plt.plot(lams, loss_list)
            plt.xlabel("Perturbation")
            plt.ylabel("Loss")
            plt.title(f"Epoch {epoch} loss landscape")
            plt.savefig(os.path.join(log_dir + "/loss_landscapes", f"landscape_epoch{epoch}.png"))
            plt.close()

        train_loss.backward()

        # if epoch%30 ==0:
        #      with torch.no_grad():
        #         # Compute total gradient norm (L2)
        #         total_grad_norm = 0.0
        #         for p in model.parameters():
        #             if p.grad is not None:
        #                 param_norm = p.grad.data.norm(2)
        #                 total_grad_norm += param_norm.item() ** 2

        #         grad_norms.append(total_grad_norm)

        optimizer.step()
        optimizer.zero_grad()


    data = {
        "log_steps": log_steps,
        "train_accuracies":train_accuracies,
        "test_accuracies":test_accuracies,
        "weight_norms": norms,
        "model_sizes": [vars(s) for s in sizes],
        "exp_freq": exp_freq,
        "total_steps": t_steps
    }

    now = datetime.now().strftime("%Y-%m-%d_%H-%M")
    tag = "1L_modadd"
    raw_string = "\n".join(["_".join(f"{k}_{v}" for k, v in vars(s).items()) for s in sizes])
    short_hash = hashlib.md5(raw_string.encode()).hexdigest()[-6:].upper()  # e.g. 'E1234A'

    # exp_name = f"{now}__{tag}__{short_hash}"
    exp_name = " | ".join(["_".join(f"{v}" for k, v in vars(s).items()) for s in sizes]) + f"@step{exp_freq}"
    exp_path = os.path.join(log_dir, exp_name)

    os.makedirs(exp_path, exist_ok=True)
    
    with open(os.path.join(exp_path, "config.json"), "w") as f:
        json.dump(data, f, indent=2)
    
    ax = plt.subplot(1, 1, 1)
    expansion_steps = [exp_freq * i for i in range(1, len(sizes))]

    plt.plot(log_steps, train_accuracies, color='red', label='train')
    plt.plot(log_steps, test_accuracies, color='green', label='test')
    

    
    if any(a>=0.95 for a in test_accuracies):
        time_to_95_pct_test = log_steps[min(i for i, acc in enumerate(test_accuracies) if acc >= 0.95)]
        plt.plot([time_to_95_pct_test]*2, [0, 1], color='green', linestyle='--')
        plt.text(time_to_95_pct_test+10, 0.65, f"@{time_to_95_pct_test} test acc\nhits 95%")

    for step in expansion_steps:
        plt.axvline(x=step, color='blue', linestyle='dotted', linewidth=1, alpha=0.2)


    plt.legend()

    plt.xlabel("Optimization Steps")
    # plt.xlim(8, 2*10**4)
    
    plt.ylim(0,1)
    ax.set_ylabel("Accuracy")
    ax2 = ax.twinx()
    ax2.set_ylabel("Weight Norm", color='purple')
    ax2.plot(log_steps, norms, color='purple', label='weight norm')
    ax2.set_ylim(27, 63)
    
    plt.xscale('log')
    plt.legend(loc=(0.015, 0.72))
    
    size_str = "dmodel,dmlp,nhead: " + " | ".join(["_".join(f"{v}" for k, v in vars(s).items()) for s in sizes])
    plt.title("1L Transformer on Modular Addition (p=113)\n"+ size_str + f"\nexp@{exp_freq} t-{t_steps}", fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(log_dir, exp_name, "plot.png"), dpi=300)
    plt.close()


In [180]:
import itertools
import os

def exp_exists(base_dir, exp_name):
    path = os.path.join(base_dir, exp_name)
    return os.path.isdir(path)

base_sizes = itertools.product([8,16], [32, 64, 128, 256], [1,2,4])
log_dir = "./log_exp@test>.4_re"

In [181]:

# exps_freq = [200, 500, 1000]

# for s, exp_freq in itertools.product(base_sizes, exps_freq):
for s in [[8,128,2]]:
    sizes = [ModelSpec(s[0],s[1],s[2]), ModelSpec(128, 512, 4)]
    exp_name = " | ".join(["_".join(f"{v}" for k, v in vars(s).items()) for s in sizes]) #+ f"@step{exp_freq}"
    if exp_exists(log_dir, exp_name):
        print(f"Experiment '{exp_name}' exists.")
    else:
        train_and_plot(sizes, 100, -1, log_dir=log_dir)

  0%|          | 0/100 [00:00<?, ?it/s]

In [183]:
for i in range(0, 100, 30):
    eigvals = np.load(f"./log_exp@test>.4_re/hessian_eigenvalues/eigvals_epoch{i}.npy")
    print(f"Epoch-{i} : {eigvals}")

Epoch-0 : [3.34205161]
Epoch-30 : [1.09845699]
Epoch-60 : [0.94881269]
Epoch-90 : [1.22867829]


In [ ]:

for s in base_sizes:
    sizes = [ModelSpec(s[0],s[1],s[2])]
    exp_name = " | ".join(["_".join(f"{v}" for k, v in vars(s).items()) for s in sizes]) # + f"@step{exp_freq}"
    if exp_exists(log_dir, exp_name):
        print(f"Experiment '{exp_name}' exists.")
    else:
        train_and_plot(sizes, 10_000, -1, log_dir=log_dir)

----

---

The L2 norm of the model's trainable parameters (viewed as a big vector in parameter space) increases when the model overfits, but then decreases when the model generalizes, finishing below the norm of the parameters at initialization! Neel Nanda observed this behavior in [this section of his notebook](https://colab.research.google.com/drive/1F6_1_cWXE5M7WocUcpQWp3v8z4b1jL20#scrollTo=BMOjY3pTjQkA), describing the decrease in weight norm as the model "cleaning up" parts of the network that were previously used to memorize, leaving behind circuits which generalize nicely.

Next we'll do something weird, which is (1) to rescale the initial model parameters down by 0.8x and (2) constrain the optimization so that weight norm is constant throughout training. We constrain optimization by, at each step of training, simply taking an optimizer (Adam) step, but then multiplying all the parameters by (initial norm / new norm), projecting us back onto the constant-norm hypersphere in parameter space.

### Constrained training run

In [ ]:
!pip install pyhessian

In [ ]:
from pyhessian import hessian

In [ ]:
seed = 42
p = 113
fraction = 0.3

In [ ]:
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
torch.set_default_dtype(torch.float64)
print(f"Using {device}")

In [ ]:
equals_token = p
x, y = torch.meshgrid(torch.arange(p), torch.arange(p), indexing='ij')
x = x.flatten()
y = y.flatten()
# plus = torch.ones(x.shape, dtype=torch.int64) * plus_token
equals = torch.ones(x.shape, dtype=torch.int64) * equals_token
prompts = torch.stack([x, y, equals], dim=1).to(device)
answers = ((x + y) % p).to(device)

data = torch.utils.data.TensorDataset(prompts, answers)
train, test = torch.utils.data.random_split(data, 
                                [int(fraction * len(data)),
                                len(data) - int(fraction * len(data))
                                ])

# model = Transformer(num_layers=1, 
#                     d_vocab=equals_token+1, 
#                     d_model=128,
#                     d_mlp=512,
#                     # d_head=32,
#                     num_heads=4,
#                     n_ctx=3, # context length
#                     act_type='ReLU', 
#                     use_cache=False, 
#                     use_ln=False # use LayerNorm
#                 ).to(device)


modelx = Transformer(num_layers=1, 
                    d_vocab=equals_token+1, 
                    d_model=128,
                    d_mlp=512,
                    d_k=32,
                    d_v=32,
                    num_heads=4,
                    n_ctx=3, # context length
                    act_type='ReLU', 
                    use_cache=False, 
                    use_ln=False # use LayerNorm
                ).to(device)

model = model.to(device=device)
print(device)

In [104]:
alpha = 0.8
with torch.no_grad():
    for param in model.parameters():
        param.data *= alpha
    norm = np.sqrt(sum(param.pow(2).sum().item() for param in model.parameters()))

In [105]:
def make_loss_fn(data, device):
    def loss_fn(model):
        return full_loss(model, data, device)

    return loss_fn

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.0, betas=(0.9, 0.98))
log_steps = []
train_losses = []
test_losses = []
train_accuracies = []
test_accuracies = []
norms = []
hessians = []


for epoch in tqdm(range(15000)):
    train_loss = full_loss(model, train, device)
    if epoch % 10 == 0:
        with torch.no_grad():
            
            log_steps.append(epoch)
            test_loss = full_loss(model, test, device)
            train_losses.append(train_loss.item())
            test_losses.append(test_loss.item())
            train_accuracies.append(full_accuracy(model, train, device))
            test_accuracies.append(full_accuracy(model, test, device))
            norms.append(np.sqrt(sum(param.pow(2).sum().item() for param in model.parameters())))

            loss_fn = make_loss_fn(train, device)
            hessian_comp = hessian(model=model, loss_fn=loss_fn, cuda=True, dataloader=None)
            
            top_eigs, _ = hessian_comp.eigenvalues(top_n=5)  
            trace = hessian_comp.trace()
            density_eigval, density_weight = hessian_comp.density()

            hessians.append({
                "epoch": epoch,
                "top_eigs": [float(ev) for ev in top_eigs],
                "trace": float(trace),
                "density_eigval": density_eigval.tolist(),
                "density_weight": density_weight.tolist()
            })

    train_loss.backward()
    optimizer.step()
    with torch.no_grad():
        new_norm = np.sqrt(sum(param.pow(2).sum().item() for param in model.parameters()))
        for param in model.parameters():
            param.data *= norm / new_norm
    optimizer.zero_grad()


In [ ]:
ax = plt.subplot(1, 1, 1)
plt.plot(log_steps, train_accuracies, color='red', label='train')
plt.plot(log_steps, test_accuracies, color='green', label='test')
plt.legend()
plt.xlabel("Optimization Steps")
plt.xlim(8, 2*10**4)
ax.set_ylabel("Accuracy")
ax2 = ax.twinx()
ax2.set_ylabel("Weight Norm", color='purple')
ax2.plot(log_steps, norms, color='purple', label='weight norm')
ax2.set_ylim(27, 63)
plt.xscale('log')
plt.legend(loc=(0.015, 0.72))
plt.title("1L Transformer on Modular Addition (p=113)\nConstrained Norm α = 0.8", fontsize=11)
plt.tight_layout()

In [ ]:
# LINEAR X-AXIS SCALE

time_to_100_pct_train = log_steps[min(i for i, acc in enumerate(train_accuracies) if acc == 1.0)]
time_to_95_pct_train = log_steps[min(i for i, acc in enumerate(train_accuracies) if acc >= 0.95)]

ax = plt.subplot(1, 1, 1)
plt.plot(log_steps, train_accuracies, color='red', label='train')
plt.plot(log_steps, test_accuracies, color='green', label='test')
plt.legend()
plt.xlabel("Optimization Steps")
plt.xlim(8, 500)

plt.plot([time_to_95_pct_train]*2, [0, 1], color='black', linestyle='dotted')
plt.text(time_to_95_pct_train+10, 0.35, "train acc\nhits 95%")

plt.plot([time_to_100_pct_train]*2, [0, 1], color='black', linestyle='dotted')
plt.text(time_to_100_pct_train+10, 0.65, "train acc\nhits 100%")
ax.set_ylabel("Accuracy")

ax2 = ax.twinx()
ax2.set_ylabel("Weight Norm", color='purple')
ax2.plot(log_steps, norms, color='purple', label='weight norm')
ax2.set_ylim(27, 63)
plt.legend(loc=(0.015, 0.72))
plt.title("1L Transformer on Modular Addition (p=113)\nConstrained Norm α = 0.8", fontsize=11)
plt.tight_layout()

 The train and test curves are still distinct, but constraining weight norm brings the train and test curves quite close together, especially relative to the gap of 1000s of steps for the baseline run! I haven't yet figured out the exactly optimal α for de-grokking training, but dropping α too low prevents the network from even fitting the train set. A higher α de-groks less effectively, bringing test loss in, just not as much.